<!-- <style> * { font-size: 100%; } </style> -->
<small> 

# rsyncd
- https://linux.die.net/man/5/rsyncd.conf
- https://superuser.com/questions/243656/how-to-configure-and-use-rsyncd


</small>

In [ ]:
## Setup Working Directory & Files

# Working Dir
WORKING_DIRECTORY="/srv/docker/rsyncd-server"
mkdir -p ${WORKING_DIRECTORY}
cd ${WORKING_DIRECTORY}

# Make Folders & Files
# mkdir -p ./{config,repositories}
# mkdir -p ./{config,secrets}
# ls -alt ${WORKING_DIRECTORY}

# Create Needed Files
# touch ${WORKING_DIRECTORY}/secrets/rsyncd.secrets

In [ ]:
# Generate User Passwords
printf 'username:%s\n' "$(openssl rand -base64 64 | head -c 64 | tr -d '\n' | tr -dc 'A-Za-z0-9')"

In [ ]:
# Generate Passwords
COUNT=12
LENGTH=64
FILTER='A-Za-z0-9' # alnum:'A-Za-z0-9'  # low:'a-z0-9'  # hex:'A-Fa-f0-9'

printf 'Password Generator | COUNT=%s | LENGTH=%s | FILTER=%s\n' \
  "$COUNT" "$LENGTH" "$FILTER"

for ((i=1; i<=COUNT; i++)); do
  openssl rand -base64 "$((LENGTH + 12))" | tr -dc 'A-Za-z0-9' | head -c "$LENGTH"
  echo
done

# Docker

## Docker Context

In [ ]:
# Use Context
# docker context use default
docker context use nas-03

## Build and Deploy Container

In [ ]:
# Build (example)
docker compose --progress plain --file docker-compose.yaml --env-file docker-compose.example.env build

In [ ]:
# Build
docker compose --progress plain --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env build

In [ ]:
# Build & Run
docker compose --progress plain --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env up --detach --build

## Rsyncd Testing

In [ ]:
# List Rsync Modules
docker exec rsyncd-server rsync rsync://127.0.0.1/

In [ ]:
# List Rsync Modules
rsync rsync://nas-03.internal

In [ ]:
# List Rsync Module Contents (example)
cat ../secrets/rsync.example.pass
chmod 600 ../secrets/rsync.example.pass
rsync --password-file=../secrets/rsync.example.pass rsync://servers_user@nas-03.internal:873/servers-main/

In [43]:
# List Rsync Module Contents (testing)
PASS_FILE_PATH="../secrets/rsync.test.pass"
# cat "${PASS_FILE_PATH}"
chmod 600 "${PASS_FILE_PATH}"
rsync --password-file=${PASS_FILE_PATH} rsync://test_user@nas-03.internal:873/testing/

drwxr-xr-x             31 2026/03/21 22:15:42 .
-rw-r--r--              6 2026/03/21 22:17:16 hello.txt


In [44]:
# Send a test file (testing)

# Password file
PASS_FILE_PATH="../secrets/rsync.test.pass"
# cat "${PASS_FILE_PATH}"
chmod 600 "${PASS_FILE_PATH}"

# File setup
TEST_FOLDER_PATH="/tmp/rsyncd-test"
mkdir --parents "${TEST_FOLDER_PATH}"
printf 'hello\n' > "${TEST_FOLDER_PATH}/hello.txt"

# Upload
rsync --archive --verbose \
  --password-file="${PASS_FILE_PATH}" \
  "${TEST_FOLDER_PATH}/" \
  "rsync://test_user@nas-03.internal:873/testing/"

sending incremental file list
hello.txt

sent 130 bytes  received 49 bytes  358.00 bytes/sec
total size is 6  speedup is 0.03


In [ ]:
# Create the Minecraft Bedrock Server & Backup Service
CONTEXT=nas-03
# docker --context lxc-docker-01 compose --profile "*" --env-file .env.lxc-docker  up --build --detach --remove-orphans

# Build the Base Image for Minecraft Bedrock Server Backup (Bedrockifier)
docker --context "${CONTEXT}" build \
  --progress plain \
  --tag local/bedrockifier-base:latest \
  --file ../repositories/minecraft-bedrock-backup/Docker/Dockerfile \
  ../repositories/minecraft-bedrock-backup

# Build + Run the Minecraft Bedrock Server & Backup Service
docker --context "${CONTEXT}" compose \
  --ansi never \
  --env-file ../.env.lxc-docker \
  up --build --detach --remove-orphans